In [39]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
import nest_asyncio

# Izinkan async di dalam Colab
nest_asyncio.apply()

# LIST URL YANG SUDAH DIBERSIHKAN (Tanpa Image, Video, & Agent)
arena_urls_clean = {
    "Text_Chat": "https://arena.ai/leaderboard/chat/text",
    "Search_Arena": "https://arena.ai/leaderboard/chat/search",
    "Vision_Arena": "https://arena.ai/leaderboard/chat/vision", # Ini LLM Multimodal (tetap punya Context/Price)
    "Document_Arena": "https://arena.ai/leaderboard/chat/document",
    "Code_WebDev": "https://arena.ai/leaderboard/code/webdev",
    "Code_Image_to_WebDev": "https://arena.ai/leaderboard/code/image-to-webdev" # Model coding
}

async def scrape_clean_leaderboards():
    all_dfs = []

    async with async_playwright() as p:
        print("🚀 Membuka Browser Playwright untuk LLM Text/Code saja...")
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
        page = await context.new_page()

        for category, url in arena_urls_clean.items():
            print(f"\n🌐 [{category}] Scraping dari: {url}")
            try:
                await page.goto(url, timeout=60000, wait_until="networkidle")
                await page.wait_for_selector("table", timeout=15000)
                await asyncio.sleep(2)

                html_content = await page.content()
                tables = pd.read_html(html_content)

                if len(tables) > 0:
                    df_temp = tables[0]
                    df_temp['Arena_Category'] = category
                    all_dfs.append(df_temp)
                    print(f"✅ Sukses! Dapet {len(df_temp)} baris data.")
                else:
                    print(f"⚠️ Tabel kosong di {category}")

            except Exception as e:
                print(f"❌ Gagal di {category}: {e}")

        await browser.close()
    return all_dfs

# Eksekusi Scraper
print("=== START CLEAN SCRAPING ===")
scraped_data_clean = asyncio.run(scrape_clean_leaderboards())

# Gabungkan dan bersihkan duplikat
if scraped_data_clean:
    df_llm = pd.concat(scraped_data_clean, ignore_index=True)

    # Langsung buang duplikat berdasarkan nama Model
    df_llm_unique = df_llm.drop_duplicates(subset=['Model'])

    print("\n" + "="*50)
    print(f"🎉 SCRAPING LLM SELESAI!")
    print(f"📊 Total Data Unik: {len(df_llm_unique)} model LLM.")
    print("="*50)

    display(df_llm_unique.head())

    # Save ke CSV baru
    df_llm_unique.to_csv("ai_arena_llm_only.csv", index=False)
    print("💾 File berhasil disimpan dengan nama: 'ai_arena_llm_only.csv'")
else:
    print("\n🚨 Gagal mengambil data.")

=== START CLEAN SCRAPING ===
🚀 Membuka Browser Playwright untuk LLM Text/Code saja...

🌐 [Text_Chat] Scraping dari: https://arena.ai/leaderboard/chat/text


/tmp/ipykernel_2612/2429124434.py:36: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html_content)


✅ Sukses! Dapet 414 baris data.

🌐 [Search_Arena] Scraping dari: https://arena.ai/leaderboard/chat/search


/tmp/ipykernel_2612/2429124434.py:36: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html_content)


✅ Sukses! Dapet 34 baris data.

🌐 [Vision_Arena] Scraping dari: https://arena.ai/leaderboard/chat/vision


...

✅ Sukses! Dapet 142 baris data.

🌐 [Code_Image_to_WebDev] Scraping dari: https://arena.ai/leaderboard/code/image-to-webdev


/tmp/ipykernel_2612/2429124434.py:36: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html_content)


✅ Sukses! Dapet 64 baris data.

🎉 SCRAPING LLM SELESAI!
📊 Total Data Unik: 499 model LLM.


,Rank,Rank Spread,Model,Score,Votes,Price $/M,Context,Arena_Category
0,1,11,gemini-4-argon-highGoogle · Proprietary,1525±9Preliminary,4892,$2 / $10,1M,Text_Chat
1,2,210,Anthropicclaude-opus-5.5-highAnthropic · Propr...,1507±8,6272,$4 / $20,1M,Text_Chat
2,3,28,Anthropicclaude-opus-4-6-highAnthropic · Propr...,1504±3,79415,$5 / $25,1M,Text_Chat
3,4,210,Anthropicclaude-fable-5-highAnthropic · Propri...,1504±4,40026,$10 / $50,1M,Text_Chat
4,5,211,Anthropicclaude-opus-4-7-highAnthropic · Propr...,1501±4,65192,$5 / $25,1M,Text_Chat


💾 File berhasil disimpan dengan nama: 'ai_arena_llm_only.csv'


In [40]:
df_llm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 863 entries, 0 to 862
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Rank            863 non-null    int64 
 1   Rank Spread     863 non-null    int64 
 2   Model           863 non-null    object
 3   Score           863 non-null    object
 4   Votes           863 non-null    int64 
 5   Price $/M       725 non-null    object
 6   Context         658 non-null    object
 7   Arena_Category  863 non-null    object
dtypes: int64(3), object(5)
memory usage: 54.1+ KB


In [41]:
# Pastikan kita pakai dataset yang udah unik (bebas duplikat)
df = df_llm_unique.copy()

# Bikin kolom 'ID' berurutan dari 1 sampai akhir, dan taruh di paling kiri (index 0)
df.insert(0, 'ID', range(1, len(df) + 1))

# Cek info dataset yang bener dan liat hasilnya
print(df.info())
display(df.head(10))

<class 'pandas.core.frame.DataFrame'>
Index: 499 entries, 0 to 798
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID              499 non-null    int64 
 1   Rank            499 non-null    int64 
 2   Rank Spread     499 non-null    int64 
 3   Model           499 non-null    object
 4   Score           499 non-null    object
 5   Votes           499 non-null    int64 
 6   Price $/M       380 non-null    object
 7   Context         335 non-null    object
 8   Arena_Category  499 non-null    object
dtypes: int64(4), object(5)
memory usage: 39.0+ KB
None


,ID,Rank,Rank Spread,Model,Score,Votes,Price $/M,Context,Arena_Category
0,1,1,11,gemini-4-argon-highGoogle · Proprietary,1525±9Preliminary,4892,$2 / $10,1M,Text_Chat
1,2,2,210,Anthropicclaude-opus-5.5-highAnthropic · Propr...,1507±8,6272,$4 / $20,1M,Text_Chat
2,3,3,28,Anthropicclaude-opus-4-6-highAnthropic · Propr...,1504±3,79415,$5 / $25,1M,Text_Chat
3,4,4,210,Anthropicclaude-fable-5-highAnthropic · Propri...,1504±4,40026,$10 / $50,1M,Text_Chat
4,5,5,211,Anthropicclaude-opus-4-7-highAnthropic · Propr...,1501±4,65192,$5 / $25,1M,Text_Chat
5,6,6,213,Anthropicclaude-fable-5.1-maxAnthropic · Propr...,1501±6,13063,$10 / $50,1M,Text_Chat
6,7,7,213,Anthropicclaude-opus-4-6Anthropic · Proprietary,1498±3,84095,$5 / $25,1M,Text_Chat
7,8,8,217,gemini-3.8-flash-highGoogle · Proprietary,1497±5Preliminary,31437,$0.75 / $3.75,1M,Text_Chat
8,9,9,323,Metamuse-spark-1.3-maxMeta · Proprietary,1494±6,14694,$1.25 / $4.25,1M,Text_Chat
9,10,10,519,Anthropicclaude-opus-4-7Anthropic · Proprietary,1494±4,66315,$5 / $25,1M,Text_Chat


In [42]:
print(df['Arena_Category'].unique())

['Text_Chat' 'Search_Arena' 'Vision_Arena' 'Code_WebDev']


In [43]:
print(df['Rank'].unique())

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 239 240 241 242 243 244 245 246 24

In [44]:
df = df.drop(columns=['Rank Spread'])
df.head()

,ID,Rank,Model,Score,Votes,Price $/M,Context,Arena_Category
0,1,1,gemini-4-argon-highGoogle · Proprietary,1525±9Preliminary,4892,$2 / $10,1M,Text_Chat
1,2,2,Anthropicclaude-opus-5.5-highAnthropic · Propr...,1507±8,6272,$4 / $20,1M,Text_Chat
2,3,3,Anthropicclaude-opus-4-6-highAnthropic · Propr...,1504±3,79415,$5 / $25,1M,Text_Chat
3,4,4,Anthropicclaude-fable-5-highAnthropic · Propri...,1504±4,40026,$10 / $50,1M,Text_Chat
4,5,5,Anthropicclaude-opus-4-7-highAnthropic · Propr...,1501±4,65192,$5 / $25,1M,Text_Chat


In [46]:
df = df.rename(columns={'Price $/M': 'Price'})
df.head()

,ID,Rank,Model,Score,Votes,Price,Context,Arena_Category
0,1,1,gemini-4-argon-highGoogle · Proprietary,1525±9Preliminary,4892,$2 / $10,1M,Text_Chat
1,2,2,Anthropicclaude-opus-5.5-highAnthropic · Propr...,1507±8,6272,$4 / $20,1M,Text_Chat
2,3,3,Anthropicclaude-opus-4-6-highAnthropic · Propr...,1504±3,79415,$5 / $25,1M,Text_Chat
3,4,4,Anthropicclaude-fable-5-highAnthropic · Propri...,1504±4,40026,$10 / $50,1M,Text_Chat
4,5,5,Anthropicclaude-opus-4-7-highAnthropic · Propr...,1501±4,65192,$5 / $25,1M,Text_Chat


In [47]:
df.to_csv('Ai_LLM_Arena.csv')
from google.colab import files
files.download('Ai_LLM_Arena.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>